In [8]:
"""
Model - hotel guest project: train the three models (with k-fold
hyperparameter tuning), validate them, refit on full data, then score the
500 Russian applicants.

Input files expected in the same folder:
    train_clean.csv   (5,000 rows: 61 features + outcome_profit,
                        outcome_damage_inc, outcome_damage_amount)
    score_clean.csv   (500 rows: 61 features, no outcomes)

Requirements:
    pip install pandas numpy scikit-learn xgboost --break-system-packages
    (xgboost is optional - the script falls back to RandomForest if missing)

Output:
    scored_applicants.csv - the 500 applicants ranked by expected_net_value

Runtime note: hyperparameter tuning runs RandomizedSearchCV with 5-fold CV,
6 times total (3 models x train-split + full-data refit). On 5,000 rows /
61 features this typically takes a few minutes. Lower TUNE_N_ITER below if
it's too slow on your machine.

CHANGE LOG:

1. safe_expm1() - clips log-space predictions before exponentiating, so a
   bad prediction can't explode into an absurd euro value.

2. Leakage check on the profit model - the population-shift diagnostic
   confirmed profit_per_night is well-populated for the score batch too
   (not blank/extrapolated), so the main model keeps it. The no-history
   comparison is still printed for visibility.

3. Calibration fix on the damage-probability model - class_weight /
   scale_pos_weight improves ranking (AUC) but skews the predicted
   probabilities away from the true base rate (this is why every one of
   the 500 applicants scored above the training base rate before the
   fix). CalibratedClassifierCV(isotonic) corrects this; verified by the
   "scored-batch mean p_damage" sanity check matching the training base
   rate.

4. Threshold sweep on the calibrated damage-probability model - only
   relevant if a hard accept/flag decision is needed; the EV formula uses
   the raw probability.

5. K-FOLD HYPERPARAMETER TUNING - n_estimators/max_depth/etc. were
   previously fixed guesses. Now RandomizedSearchCV with 5-fold CV picks
   them for each of the three models, both on the train split (for honest
   validation metrics) and again on the full dataset (for the production
   models used to score the 500 applicants).
"""

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    roc_auc_score, precision_score, recall_score, f1_score, brier_score_loss,
)

try:
    from xgboost import XGBRegressor, XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not installed - falling back to RandomForest only.")
    print("Run: pip install xgboost --break-system-packages\n")

RANDOM_SEED = 42
EXPM1_CAP = 11.5      # clip log-space predictions here before expm1 (~ e^11.5-1 = ~98,700 euros)
TUNE_N_ITER = 20       # RandomizedSearchCV candidates for profit / damage-probability models
TUNE_N_ITER_SMALL = 15 # fewer candidates for the small damage-amount subset (~1,277 rows)
TUNE_CV = 5            # k-fold CV used inside every hyperparameter search


def safe_expm1(log_values):
    """expm1 with a clip so a runaway log-space prediction can't blow up
    into an absurd euro value once exponentiated."""
    return np.expm1(np.clip(log_values, a_min=None, a_max=EXPM1_CAP))


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def trivial_brier(y_true):
    """Brier score of just predicting the base rate for every row - the
    bar any real probability model should beat."""
    p = np.mean(y_true)
    return p * (1 - p)


def report_regression(name, y_true, y_pred):
    print(f"{name:26s} RMSE={rmse(y_true, y_pred):8.2f}  "
          f"MAE={mean_absolute_error(y_true, y_pred):8.2f}  "
          f"R2={r2_score(y_true, y_pred):.3f}")


def report_classification(name, y_true, proba, threshold=0.5):
    pred_label = (proba >= threshold).astype(int)
    print(f"{name:26s} AUC={roc_auc_score(y_true, proba):.3f}  "
          f"Brier={brier_score_loss(y_true, proba):.3f}  "
          f"Precision={precision_score(y_true, pred_label, zero_division=0):.3f}  "
          f"Recall={recall_score(y_true, pred_label, zero_division=0):.3f}  "
          f"MeanProba={np.mean(proba):.3f}")


def make_regressor(n_estimators, max_depth, learning_rate=0.05):
    """Fixed-hyperparameter regressor - used only for the quick leakage-check
    comparison model, where tuning isn't the point."""
    if HAS_XGB:
        return XGBRegressor(n_estimators=n_estimators, max_depth=max_depth,
                             learning_rate=learning_rate, subsample=0.8,
                             colsample_bytree=0.8, random_state=RANDOM_SEED, n_jobs=1)
    return RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth + 4,
                                  random_state=RANDOM_SEED, n_jobs=-1)


def tune_regressor(X, y, scoring="neg_root_mean_squared_error",
                    n_iter=TUNE_N_ITER, cv=TUNE_CV, label=""):
    """RandomizedSearchCV with k-fold CV to choose hyperparameters, instead
    of using fixed n_estimators/max_depth guesses. Returns the best
    estimator, already refit on the full (X, y) passed in."""
    if HAS_XGB:
        base = XGBRegressor(random_state=RANDOM_SEED, n_jobs=-1)
        param_dist = {
            "n_estimators": [100, 150, 200, 300, 400, 500],
            "max_depth": [2, 3, 4, 5, 6],
            "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08, 0.1],
            "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
            "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
        }
    else:
        base = RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=1)
        param_dist = {
            "n_estimators": [100, 200, 300, 400, 500],
            "max_depth": [4, 6, 8, 10, 12, None],
            "min_samples_leaf": [1, 2, 4, 8],
        }
    # n_jobs=1 here on purpose: RandomizedSearchCV's own parallelism spawns
    # new Python processes (via joblib/loky), which re-import numpy/sklearn
    # from scratch. On environments with two conflicting numpy installs
    # (e.g. a conda env plus a "pip install --user" package pulling in a
    # newer numpy) those subprocess re-imports crash with a
    # BrokenProcessPool/ImportError even though the main process is fine.
    # XGBoost still uses multiple CPU cores internally via n_jobs=-1 above -
    # that's thread-based (OpenMP), not a new process, so it's unaffected.
    search = RandomizedSearchCV(base, param_dist, n_iter=n_iter, cv=cv, scoring=scoring,
                                 random_state=RANDOM_SEED, n_jobs=1)
    search.fit(X, y)
    print(f"  [{label}] best params: {search.best_params_}")
    print(f"  [{label}] best {cv}-fold CV score ({scoring}): {search.best_score_:.4f}")
    return search.best_estimator_


def tune_classifier(X, y, scale_pos_weight, scoring="roc_auc",
                     n_iter=TUNE_N_ITER, cv=TUNE_CV, label=""):
    """Same idea as tune_regressor, for the damage-probability classifier.
    Tunes on roc_auc (ranking ability) - calibration is handled separately
    afterward via CalibratedClassifierCV."""
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=RANDOM_SEED)
    if HAS_XGB:
        base = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=RANDOM_SEED, n_jobs=-1)
        param_dist = {
            "n_estimators": [100, 150, 200, 300, 400, 500],
            "max_depth": [2, 3, 4, 5, 6],
            "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08, 0.1],
            "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
            "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
        }
    else:
        base = RandomForestClassifier(class_weight="balanced", random_state=RANDOM_SEED, n_jobs=1)
        param_dist = {
            "n_estimators": [100, 200, 300, 400, 500],
            "max_depth": [4, 6, 8, 10, 12, None],
            "min_samples_leaf": [1, 2, 4, 8],
        }
    # n_jobs=1 here too - see comment in tune_regressor above.
    search = RandomizedSearchCV(base, param_dist, n_iter=n_iter, cv=skf, scoring=scoring,
                                 random_state=RANDOM_SEED, n_jobs=1)
    search.fit(X, y)
    print(f"  [{label}] best params: {search.best_params_}")
    print(f"  [{label}] best {cv}-fold CV score ({scoring}): {search.best_score_:.4f}")
    return search.best_estimator_


# ---------------------------------------------------------------------------
# 0. Load the cleaned data
# ---------------------------------------------------------------------------
train_clean = pd.read_csv("train_clean.csv")
score_clean = pd.read_csv("score_clean.csv")

OUTCOME_COLS = ["outcome_profit", "outcome_damage_inc", "outcome_damage_amount"]
feature_cols = [c for c in train_clean.columns if c not in OUTCOME_COLS]

X = train_clean[feature_cols]
y = train_clean[OUTCOME_COLS]

assert list(score_clean.columns) == feature_cols, "score_clean columns must match train features"
X_score = score_clean[feature_cols]

print(f"Train: {X.shape}, Score: {X_score.shape}")

# ---------------------------------------------------------------------------
# 1. Train / validation split (stratified on damage incident)
# ---------------------------------------------------------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y["outcome_damage_inc"]
)
print(f"Train split: {X_train.shape}, Val split: {X_val.shape}\n")

# ===========================================================================
# MODEL 1 - Expected Profit (regression, all guests)
# ===========================================================================
y_profit_train_log = np.log1p(y_train["outcome_profit"])
y_val_profit_actual = y_val["outcome_profit"].values

lr_profit = LinearRegression()
lr_profit.fit(X_train, y_profit_train_log)
pred_profit_lr = safe_expm1(lr_profit.predict(X_val))

print("Tuning profit model (RandomizedSearchCV, 5-fold CV)...")
profit_model = tune_regressor(X_train, y_profit_train_log, label="profit/train-split")
pred_profit_main = safe_expm1(profit_model.predict(X_val))

print("\n--- Profit model validation (euros) ---")
report_regression("Linear (baseline)", y_val_profit_actual, pred_profit_lr)
report_regression("Main model", y_val_profit_actual, pred_profit_main)

# Leakage check (informational, fixed hyperparameters - diagnostic confirmed
# profit_per_night is well-populated for the score batch too, so the main
# tuned model above is kept as-is for production).
leak_cols = [c for c in ["profit_per_night", "profit_am", "profit_am_log"] if c in feature_cols]
reduced_cols = [c for c in feature_cols if c not in leak_cols]

print(f"\n--- Leakage check: profit model without {leak_cols} ---")
profit_model_noleak = make_regressor(n_estimators=300, max_depth=4)
profit_model_noleak.fit(X_train[reduced_cols], y_profit_train_log)
pred_profit_noleak = safe_expm1(profit_model_noleak.predict(X_val[reduced_cols]))
report_regression("Main, w/o profit hist.", y_val_profit_actual, pred_profit_noleak)
print("(Diagnostic confirmed profit_per_night is populated similarly in the score")
print("batch as in training, so this drop reflects a strong real feature, not")
print("extrapolation on missing data. Tuned main model with profit_per_night is used.)")

# ===========================================================================
# MODEL 2 - Damage Probability (classification, all guests)
# ===========================================================================
y_dmg_train = y_train["outcome_damage_inc"]
y_dmg_val = y_val["outcome_damage_inc"]
base_rate_train = y_dmg_train.mean()

logit = LogisticRegression(max_iter=1000, class_weight="balanced")
logit.fit(X_train, y_dmg_train)
proba_dmg_lr = logit.predict_proba(X_val)[:, 1]

pos, neg = y_dmg_train.sum(), len(y_dmg_train) - y_dmg_train.sum()
print("\nTuning damage probability model (RandomizedSearchCV, 5-fold CV)...")
damage_model_raw = tune_classifier(X_train, y_dmg_train, scale_pos_weight=neg / pos,
                                    label="damage/train-split")
proba_dmg_raw = damage_model_raw.predict_proba(X_val)[:, 1]

print(f"\n--- Damage probability model validation (training base rate = {base_rate_train:.3f}) ---")
print(f"Trivial Brier (predict base rate for everyone): {trivial_brier(y_dmg_val):.3f}")
report_classification("Logistic (baseline)", y_dmg_val, proba_dmg_lr)
report_classification("Main model, raw", y_dmg_val, proba_dmg_raw)

# Calibration fix - see CHANGE LOG item 3. Reuses the tuned hyperparameters
# (via clone) but CalibratedClassifierCV refits from scratch across its own
# 5-fold split to produce calibrated probabilities.
damage_model_cal = CalibratedClassifierCV(clone(damage_model_raw), method="isotonic", cv=5)
damage_model_cal.fit(X_train, y_dmg_train)
proba_dmg_main = damage_model_cal.predict_proba(X_val)[:, 1]

report_classification("Main model, calibrated", y_dmg_val, proba_dmg_main)
if brier_score_loss(y_dmg_val, proba_dmg_main) < trivial_brier(y_dmg_val):
    print("Calibrated Brier beats the trivial baseline - probabilities are usable now.")
else:
    print("WARNING: calibrated Brier still doesn't beat the trivial baseline - "
          "investigate further before trusting p_damage in the EV formula.")

print("\n--- Damage probability: threshold sweep (calibrated model) ---")
best_f1, best_thresh = -1, 0.5
for t in np.arange(0.10, 0.60, 0.05):
    pred_label = (proba_dmg_main >= t).astype(int)
    p = precision_score(y_dmg_val, pred_label, zero_division=0)
    r = recall_score(y_dmg_val, pred_label, zero_division=0)
    f1 = f1_score(y_dmg_val, pred_label, zero_division=0)
    print(f"  threshold={t:.2f}  precision={p:.3f}  recall={r:.3f}  f1={f1:.3f}")
    if f1 > best_f1:
        best_f1, best_thresh = f1, t
print(f"Best F1 at threshold={best_thresh:.2f} (F1={best_f1:.3f})")
print("(This only matters for a hard accept/flag decision - the EV formula")
print("below uses the raw calibrated probability, not a thresholded label.)")

# ===========================================================================
# MODEL 3 - Damage Amount (regression, damage cases only)
# ===========================================================================
mask_train = y_train["outcome_damage_inc"] == 1
mask_val = y_val["outcome_damage_inc"] == 1

X_dmg_train, X_dmg_val = X_train[mask_train], X_val[mask_val]
y_dmg_amt_train_log = np.log1p(y_train.loc[mask_train, "outcome_damage_amount"])
y_dmg_amt_val_actual = y_val.loc[mask_val, "outcome_damage_amount"].values

print(f"\nDamage-amount rows -> train: {X_dmg_train.shape[0]}, val: {X_dmg_val.shape[0]}")

lr_amt = LinearRegression()
lr_amt.fit(X_dmg_train, y_dmg_amt_train_log)
pred_amt_lr = safe_expm1(lr_amt.predict(X_dmg_val))

print("Tuning damage amount model (RandomizedSearchCV, 5-fold CV)...")
amount_model = tune_regressor(X_dmg_train, y_dmg_amt_train_log,
                               n_iter=TUNE_N_ITER_SMALL, label="damage-amount/train-split")
pred_amt_main = safe_expm1(amount_model.predict(X_dmg_val))

print("\n--- Damage amount model validation (euros, damage cases only) ---")
report_regression("Linear (baseline)", y_dmg_amt_val_actual, pred_amt_lr)
report_regression("Main model", y_dmg_amt_val_actual, pred_amt_main)

# This subset is small (~1,275 rows total), so also run 5-fold CV on every
# damage case for a more stable read on performance. Reuses the hyperparameters
# found above (via clone) rather than re-tuning inside every fold.
dmg_mask_all = y["outcome_damage_inc"] == 1
X_dmg_all = X[dmg_mask_all]
y_dmg_amt_all_log = np.log1p(y.loc[dmg_mask_all, "outcome_damage_amount"])

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_scores = []
for fold, (tr_idx, te_idx) in enumerate(kf.split(X_dmg_all), 1):
    m = clone(amount_model)
    m.fit(X_dmg_all.iloc[tr_idx], y_dmg_amt_all_log.iloc[tr_idx])
    pred = safe_expm1(m.predict(X_dmg_all.iloc[te_idx]))
    actual = np.expm1(y_dmg_amt_all_log.iloc[te_idx])
    fold_rmse = rmse(actual, pred)
    cv_scores.append(fold_rmse)
    print(f"  fold {fold}: RMSE = {fold_rmse:.2f}")
print(f"5-fold CV RMSE: {np.mean(cv_scores):.2f} +/- {np.std(cv_scores):.2f}")
print("(R2 here is weak - treat expected_damage_amount as a rough estimate, not a precise figure.)")

# ===========================================================================
# REFIT chosen models on ALL eligible training data (re-tuned on full data)
# ===========================================================================
print("\nTuning final profit model on all data (RandomizedSearchCV, 5-fold CV)...")
final_profit_model = tune_regressor(X, np.log1p(y["outcome_profit"]), label="profit/final-full-data")

pos_all, neg_all = y["outcome_damage_inc"].sum(), len(y) - y["outcome_damage_inc"].sum()
print("\nTuning final damage probability model on all data (RandomizedSearchCV, 5-fold CV)...")
damage_model_final_raw = tune_classifier(X, y["outcome_damage_inc"], scale_pos_weight=neg_all / pos_all,
                                          label="damage/final-full-data")
final_damage_model = CalibratedClassifierCV(clone(damage_model_final_raw), method="isotonic", cv=5)
final_damage_model.fit(X, y["outcome_damage_inc"])

print("\nTuning final damage amount model on all data (RandomizedSearchCV, 5-fold CV)...")
final_amount_model = tune_regressor(X_dmg_all, y_dmg_amt_all_log,
                                     n_iter=TUNE_N_ITER_SMALL, label="damage-amount/final-full-data")

print("\nFinal models refit on all eligible training data (damage model calibrated).")

# ===========================================================================
# SCORE the 500 Russian applicants
# ===========================================================================
expected_profit = safe_expm1(final_profit_model.predict(X_score))
p_damage = final_damage_model.predict_proba(X_score)[:, 1]
expected_damage_amount = safe_expm1(final_amount_model.predict(X_score))

expected_damage_cost = p_damage * expected_damage_amount
expected_net_value = expected_profit - expected_damage_cost

results = score_clean.copy()
results["expected_profit"] = expected_profit
results["p_damage"] = p_damage
results["expected_damage_amount"] = expected_damage_amount
results["expected_damage_cost"] = expected_damage_cost
results["expected_net_value"] = expected_net_value
results = results.sort_values("expected_net_value", ascending=False).reset_index(drop=True)

results.to_csv("scored_applicants.csv", index=False)
print(f"\nScored {len(results)} applicants -> scored_applicants.csv")
print(results[["expected_profit", "p_damage", "expected_damage_amount",
                "expected_damage_cost", "expected_net_value"]].describe())

# ===========================================================================
# SANITY CHECKS
# ===========================================================================
print("\n--- Sanity checks ---")
print("Any negative expected_profit predictions?", bool((expected_profit < 0).any()))
print(f"Training base rate for damage: {y['outcome_damage_inc'].mean():.3f}")
print(f"Scored-batch mean p_damage:    {p_damage.mean():.3f}  "
      f"(should be close to the training base rate)")
print("p_damage range:", round(p_damage.min(), 3), "-", round(p_damage.max(), 3))

importances = pd.Series(final_profit_model.feature_importances_, index=feature_cols)
print("\nTop 10 features driving expected profit:")
print(importances.sort_values(ascending=False).head(10))

print("\nTop 5 applicants by expected net value:")
print(results.head()[["expected_profit", "p_damage", "expected_damage_amount", "expected_net_value"]])
print("\nBottom 5 applicants by expected net value:")
print(results.tail()[["expected_profit", "p_damage", "expected_damage_amount", "expected_net_value"]])

Train: (5000, 61), Score: (500, 61)
Train split: (4000, 61), Val split: (1000, 61)

Tuning profit model (RandomizedSearchCV, 5-fold CV)...
  [profit/train-split] best params: {'subsample': 1.0, 'n_estimators': 300, 'max_depth': 2, 'learning_rate': 0.1, 'colsample_bytree': 0.6}
  [profit/train-split] best 5-fold CV score (neg_root_mean_squared_error): -0.2519

--- Profit model validation (euros) ---
Linear (baseline)          RMSE=  704.43  MAE=  433.26  R2=0.762
Main model                 RMSE=  621.84  MAE=  386.67  R2=0.815

--- Leakage check: profit model without ['profit_per_night', 'profit_am', 'profit_am_log'] ---
Main, w/o profit hist.     RMSE=  926.90  MAE=  442.05  R2=0.588
(Diagnostic confirmed profit_per_night is populated similarly in the score
batch as in training, so this drop reflects a strong real feature, not
extrapolation on missing data. Tuned main model with profit_per_night is used.)

Tuning damage probability model (RandomizedSearchCV, 5-fold CV)...
  [damage/tra